# Trabajo Práctico Final: Airbnb Price
# Preprocesamiento y manipulación de datos

En esta notebook se realiza la limpieza, transformación y preparación del dataset Airbnb, para obtener un conjunto de datos adecuado para la posterior aplicación de modelos de Machine Learning.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("airbnb_us.csv")

print("Dimensiones originales:", df.shape)
df.head()

Dimensiones originales: (19309, 29)


,id,property_type,room_type,amenities,accommodates,bathrooms,bed_type,cancellation_policy,cleaning_fee,city,...,longitude,name,neighbourhood,number_of_reviews,review_scores_rating,thumbnail_url,zipcode,bedrooms,beds,price
0,13418779,House,Entire home/apt,"{TV,""Cable TV"",Internet,""Wireless Internet"",Ki...",4,1.0,Real Bed,flexible,True,SF,...,-122.431619,Beautiful Flat in the Heart of SF!,Lower Haight,0,NaN,https://a0.muscache.com/im/pictures/72208dad-9...,94117.0,2.0,2.0,750.0
1,3808709,Apartment,Entire home/apt,"{TV,Internet,""Wireless Internet"",""Air conditio...",2,1.0,Real Bed,moderate,True,DC,...,-77.034596,Great studio in midtown DC,Columbia Heights,4,40.0,NaN,20009,0.0,1.0,115.0
2,12422935,Apartment,Private room,"{TV,""Wireless Internet"",Heating,""Smoke detecto...",2,1.0,Real Bed,strict,True,SF,...,-122.429526,Comfort Suite San Francisco,Noe Valley,3,100.0,https://a0.muscache.com/im/pictures/82509143-4...,94131,1.0,1.0,85.0
3,180792,House,Private room,"{TV,""Cable TV"",""Wireless Internet"",""Pets live ...",2,1.0,Real Bed,moderate,True,SF,...,-122.501095,Cozy Garden Studio - Private Entry,Richmond District,159,99.0,https://a0.muscache.com/im/pictures/0ed6c128-7...,94121,1.0,1.0,120.0
4,2658946,Apartment,Entire home/apt,"{TV,""Cable TV"",Internet,""Wireless Internet"",""A...",6,1.5,Real Bed,strict,True,DC,...,-77.031189,Charming 2 bdrm in trendy U/14th streets w/par...,U Street Corridor,13,89.0,NaN,20009,2.0,3.0,200.0


### **Eliminación de registros duplicados**

Se eliminan las publicaciones duplicadas utilizando el identificador único de cada alojamiento `id`.


In [2]:
duplicados = df.duplicated(subset='id').sum()

print("Registros duplicados:", duplicados)

df = df.drop_duplicates(subset='id')

print("Dimensiones luego de eliminar duplicados:", df.shape)

Registros duplicados: 0
Dimensiones luego de eliminar duplicados: (19309, 29)


### **Tratamiento de valores atípicos**

Se filtra la variable objetivo `price`, conservando los registros comprendidos entre los percentiles 1 y 99. De esta manera, se reducen los efectos de los valores extremos sobre el entrenamiento de los modelos.

In [3]:
p1, p99 = df['price'].quantile([0.01, 0.99])

print("Percentil 1:", p1)
print("Percentil 99:", p99)

df = df[
    (df['price'] >= p1) &
    (df['price'] <= p99)
].copy()

print("Dimensiones luego del filtrado:", df.shape)

Percentil 1: 28.99999999999999
Percentil 99: 1100.0000000000016
Dimensiones luego del filtrado: (18949, 29)


### **Tratamiento de valores anómalos**

Se consideran anómalos los registros que presentan cero baños o cero camas. 
Estos valores se reemplazan por datos faltantes (NaN), que posteriormente serán tratados mediante técnicas de imputación.

In [4]:
df.loc[df['bathrooms'] == 0, 'bathrooms'] = np.nan

df.loc[df['beds'] == 0, 'beds'] = np.nan

print("Valores nulos en bathrooms:", df['bathrooms'].isnull().sum())
print("Valores nulos en beds:", df['beds'].isnull().sum())

Valores nulos en bathrooms: 80
Valores nulos en beds: 26


### **Creación de nuevas variables**

A partir de las variables originales se generan dos nuevas características:

- num_amenities: cantidad de servicios y comodidades ofrecidos por la propiedad.
- antiguedad_host_dias: antigüedad del anfitrión en la plataforma, expresada en días.

In [5]:
df['num_amenities'] = df['amenities'].apply(
    lambda s: 0 if s == '{}' else s.count(',') + 1
)

df['host_since'] = pd.to_datetime(
    df['host_since'],
    errors='coerce'
)

fecha_maxima = df['host_since'].max()

df['antiguedad_host_dias'] = (
    fecha_maxima - df['host_since']
).dt.days

print(df[['num_amenities', 'antiguedad_host_dias']].head())

   num_amenities  antiguedad_host_dias
0             15                 899.0
1             12                 948.0
2             10                 119.0
3             21                 853.0
4             25                1818.0


### **Transformación de variables**

La variable `host_response_rate` se encuentra expresada como texto y contiene el símbolo %.
Se transforma a formato numérico para poder utilizarla en los modelos predictivos.

In [7]:
df['host_response_rate'] = (
    df['host_response_rate']
    .str.replace('%', '', regex=False)
    .astype(float)
)

print(df['host_response_rate'].head())

0      NaN
1    100.0
2    100.0
3    100.0
4    100.0
Name: host_response_rate, dtype: float64


### **Verificaciones**

Se verifican las dimensiones finales, los tipos de datos y la presencia de valores faltantes para comprobar que las manipulaciones se hayan aplicado correctamente y que los datos esten en condiciones para ser utilizados en el entrenamiento y evaluación de los modelos.

In [10]:
print("Dimensiones finales:", df.shape)

print("\nTipos de datos:")
print(df.dtypes)

print("\nCantidad de valores nulos:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

Dimensiones finales: (18949, 31)

Tipos de datos:
id                                 int64
property_type                     object
room_type                         object
amenities                         object
accommodates                       int64
bathrooms                        float64
bed_type                          object
cancellation_policy               object
cleaning_fee                        bool
city                              object
description                       object
first_review                      object
host_has_profile_pic              object
host_identity_verified            object
host_response_rate               float64
host_since                datetime64[ns]
instant_bookable                  object
last_review                       object
latitude                         float64
longitude                        float64
name                              object
neighbourhood                     object
number_of_reviews                  int64
review_

### **Se verifica que la tabla final haya quedado de forma correcta**

In [11]:
df.head()

,id,property_type,room_type,amenities,accommodates,bathrooms,bed_type,cancellation_policy,cleaning_fee,city,...,neighbourhood,number_of_reviews,review_scores_rating,thumbnail_url,zipcode,bedrooms,beds,price,num_amenities,antiguedad_host_dias
0,13418779,House,Entire home/apt,"{TV,""Cable TV"",Internet,""Wireless Internet"",Ki...",4,1.0,Real Bed,flexible,True,SF,...,Lower Haight,0,NaN,https://a0.muscache.com/im/pictures/72208dad-9...,94117.0,2.0,2.0,750.0,15,899.0
1,3808709,Apartment,Entire home/apt,"{TV,Internet,""Wireless Internet"",""Air conditio...",2,1.0,Real Bed,moderate,True,DC,...,Columbia Heights,4,40.0,NaN,20009,0.0,1.0,115.0,12,948.0
2,12422935,Apartment,Private room,"{TV,""Wireless Internet"",Heating,""Smoke detecto...",2,1.0,Real Bed,strict,True,SF,...,Noe Valley,3,100.0,https://a0.muscache.com/im/pictures/82509143-4...,94131,1.0,1.0,85.0,10,119.0
3,180792,House,Private room,"{TV,""Cable TV"",""Wireless Internet"",""Pets live ...",2,1.0,Real Bed,moderate,True,SF,...,Richmond District,159,99.0,https://a0.muscache.com/im/pictures/0ed6c128-7...,94121,1.0,1.0,120.0,21,853.0
4,2658946,Apartment,Entire home/apt,"{TV,""Cable TV"",Internet,""Wireless Internet"",""A...",6,1.5,Real Bed,strict,True,DC,...,U Street Corridor,13,89.0,NaN,20009,2.0,3.0,200.0,25,1818.0
